# Multi-Agent AI Analyst — Colab runner

The zero-install, zero-credit-card path to a **live public link** (F14).

A supervisor routes your question to specialists (documents, web, SQL, code); a critic verifies the answer before you see it.

**What you need:** a free Gemini key from https://aistudio.google.com/apikey — that is the only required one.

Run the cells top to bottom. The last cell prints a public `*.gradio.live` URL that stays up for ~72 hours.


## 1 · Clone the repository

Replace the URL with your own fork.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/multi-agent-ai-analyst.git"  # <-- change me

import os, pathlib
if not pathlib.Path("multi-agent-ai-analyst").exists():
    !git clone -q $REPO_URL multi-agent-ai-analyst
%cd multi-agent-ai-analyst/backend
!ls

## 2 · Install dependencies

Takes 2–3 minutes. The restart warning at the end is normal and safe to ignore.

In [ ]:
!pip install -q -r requirements.txt gradio
print("done")

## 3 · Add your keys

Colab's secrets manager (the 🔑 icon in the left sidebar) is the safe way — the key is never written into the notebook and never lands in your git history.

Add a secret named `GOOGLE_API_KEY`, toggle notebook access on, then run this cell.

Optional secrets: `TAVILY_API_KEY` (web search), `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` (tracing). Skip them and the system still works.

In [ ]:
import os

try:
    from google.colab import userdata
    def secret(name):
        try:
            return userdata.get(name) or ""
        except Exception:
            return ""
except ImportError:
    def secret(name):
        return os.environ.get(name, "")

for key in ["GOOGLE_API_KEY", "TAVILY_API_KEY",
            "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]:
    value = secret(key)
    if value:
        os.environ[key] = value

# Fallback: prompt for the one required key if the secret is not set.
if not os.environ.get("GOOGLE_API_KEY"):
    from getpass import getpass
    os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY: ")

os.environ.setdefault("GEMINI_MODEL", "gemini-2.5-flash")
os.environ.setdefault("GEMINI_EMBED_MODEL", "models/gemini-embedding-001")
os.environ.setdefault("EMBED_DIM", "768")
os.environ.setdefault("QDRANT_PATH", "./data/qdrant")
os.environ.setdefault("SQLITE_PATH", "./data/company.db")

from app.config import settings
print(settings.capability_report())

## 4 · Build the data

Seeds the SQLite database (deterministic) and embeds the document corpus into Qdrant.

In [ ]:
!python -m ingestion.seed_db

In [ ]:
!python -m ingestion.ingest --reset

## 5 · Prove every feature works

One line per rubric criterion. Anything needing a key you have not set reports SKIP, not FAIL.

In [ ]:
!python -m scripts.smoke

## 6 · Ask one question from the CLI

Watch the trace: supervisor → data → retriever → generate → critic.

In [ ]:
!python -m app.graph "How many customers churned in Q2 2026, and why did they leave?" 

## 7 · Launch the public UI  ← this is the F14 deliverable

`share=True` returns a public `*.gradio.live` URL. Copy it into your submission.

Keep this cell running — the link dies when the cell stops.

In [ ]:
import app_gradio
app_gradio.build_ui().queue().launch(share=True, debug=False)

## 8 · (Optional) Run the evaluation harness

Scores the fixed test set with an LLM judge, an exact-fact check and RAGAS — **with and without the critic**, so the critic's value is measured rather than asserted.

Stop the Gradio cell first: embedded Qdrant is single-process.

`--quick` runs 6 of the 14 cases, which is gentler on the free-tier rate limit.

In [ ]:
!python -m eval.run_eval --quick --no-ragas

In [ ]:
# The full run, including RAGAS. Slower and more quota-hungry.
# !python -m eval.run_eval

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

latest = Path("eval/results/latest.md")
display(Markdown(latest.read_text())) if latest.exists() else print("Run the evaluation first.")

---

### Troubleshooting

| Symptom | Fix |
|---|---|
| `GOOGLE_API_KEY is not set` | Re-run cell 3; check the secret's notebook-access toggle |
| `Storage folder … already accessed` | Qdrant embedded is single-process — stop the Gradio cell first |
| `429 / quota exceeded` | Free-tier rate limit. Wait a minute, or use `--quick` |
| `model not found` | Set `GEMINI_MODEL` in cell 3 to a currently available model |
| Share link expired | Re-run cell 7 — links last ~72 h |
